# Sheep Breed Classification — Exploratory Data Analysis

This notebook visualises the dataset before training.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from collections import Counter

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

PROCESSED_DIR = Path('../') / cfg['data']['processed_dir']
CLASSES = cfg['classes']

## 1. Class Distribution

In [ ]:
split_counts = {}
for split in ['train', 'val', 'test']:
    counts = {}
    for cls in CLASSES:
        cls_dir = PROCESSED_DIR / split / cls
        if cls_dir.exists():
            counts[cls] = len(list(cls_dir.glob('*.*')))
        else:
            counts[cls] = 0
    split_counts[split] = counts

x = np.arange(len(CLASSES))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

for i, (split, color) in enumerate(zip(['train','val','test'], ['steelblue','orange','green'])):
    vals = [split_counts[split][c] for c in CLASSES]
    ax.bar(x + i*width, vals, width, label=split, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_ylabel('Image count')
ax.set_title('Class Distribution per Split')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

print('\nTotal images per split:')
for split in ['train','val','test']:
    print(f'  {split}: {sum(split_counts[split].values())}')

## 2. Sample Images per Class

In [ ]:
N_SAMPLES = 4
fig, axes = plt.subplots(len(CLASSES), N_SAMPLES, figsize=(N_SAMPLES * 3, len(CLASSES) * 3))

for row, cls in enumerate(CLASSES):
    cls_dir = PROCESSED_DIR / 'train' / cls
    imgs = sorted(cls_dir.glob('*.*'))[:N_SAMPLES] if cls_dir.exists() else []
    for col in range(N_SAMPLES):
        ax = axes[row, col]
        if col < len(imgs):
            ax.imshow(Image.open(imgs[col]))
        else:
            ax.axis('off')
            continue
        ax.set_title(cls if col == 0 else '', fontsize=10)
        ax.axis('off')

plt.suptitle('Sample Images per Breed (train set)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Pixel Intensity Statistics

In [ ]:
means, stds = [], []
for cls in CLASSES:
    cls_dir = PROCESSED_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    for p in list(cls_dir.glob('*.*'))[:50]:  # sample 50 per class
        arr = np.array(Image.open(p).convert('RGB')) / 255.0
        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))

means = np.array(means)
stds  = np.array(stds)
channels = ['R', 'G', 'B']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ch in enumerate(channels):
    axes[0].hist(means[:, i], bins=30, alpha=0.6, label=ch, color=['red','green','blue'][i])
    axes[1].hist(stds[:, i],  bins=30, alpha=0.6, label=ch, color=['red','green','blue'][i])

axes[0].set_title('Channel Mean Distribution')
axes[1].set_title('Channel Std Distribution')
for ax in axes:
    ax.legend()
    ax.set_xlabel('Value')

plt.tight_layout()
plt.savefig('../results/figures/pixel_stats.png', dpi=150)
plt.show()

print('Dataset mean (RGB):', means.mean(axis=0).round(4))
print('Dataset std  (RGB):', stds.mean(axis=0).round(4))

## 4. Augmentation Preview

In [ ]:
from dataset import get_transforms
import torch

# Pick one image
sample_path = next((PROCESSED_DIR / 'train' / CLASSES[0]).glob('*.*'))
raw = np.array(Image.open(sample_path).convert('RGB'))

transform = get_transforms('train', cfg['data']['image_size'], cfg)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(raw)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 10):
    aug = transform(image=raw)['image']
    # Unnormalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    display = (aug * std + mean).clamp(0,1).permute(1,2,0).numpy()
    r, c = divmod(i, 5)
    axes[r, c].imshow(display)
    axes[r, c].set_title(f'Aug {i}')
    axes[r, c].axis('off')

plt.suptitle('Augmentation Samples', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/augmentation_preview.png', dpi=120)
plt.show()